# NB 02 · State contract

**Series:** Learning LangGraph through literate notebooks, notebook 2 of 7.
**Central question:** how do several steps share one consistent state, and who may change what?

In NB 01 the graph had one node, and that node wrote one field. The research assistant needs
several steps: some collect documents, one rates the evidence, one writes the answer. They all
work on the same state. This notebook gives that state a fixed shape, rules for combining what
different nodes write, and a contract that says which node may write which field.

## Learning goals

After this notebook you can:

- design a state schema with `TypedDict` and explain what each field is for;
- explain LangGraph's default rule for updates, and when it loses data without any warning;
- write a reducer and attach it to a field with `Annotated`;
- run two nodes in parallel and merge their results so that every run gives the same state;
- write a state contract and check it against what the nodes actually did.

## Setup

This notebook makes no model calls, so it runs the same in every `LLM_MODE`. It loads documents from
the corpus in `data/corpus/` through two helpers from the package: `load_corpus()` reads the Markdown
files, and `to_source_doc()` turns one of them into the dictionary that travels in the state.

In [ ]:
import operator
import sys
from importlib.metadata import version
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langgraph.graph import END, START, StateGraph

from research_assistant.hello import QUESTION
from research_assistant.state import SourceDoc
from research_assistant.tools import load_corpus, to_source_doc

load_dotenv()

assert sys.version_info >= (3, 12), "This series needs Python 3.12 or newer."
print(f"Python {sys.version.split()[0]}")
for package in ("langgraph", "langchain-core"):
    print(f"{package} {version(package)}")
print("This notebook makes no model calls.")

## Concept: the state is a shared form

Think of the state as a form with fixed fields that is handed from node to node. Every node can
read the whole form. When it is done, it does not hand back a new form: it returns only the fields
it filled in, and LangGraph copies them onto the form.

| Field | What it holds | Written by |
|---|---|---|
| `question` | the question asked | the caller, before the run |
| `documents` | the documents found so far | the collecting nodes |
| `evidence_score` | how strong the evidence is | the scoring node only |
| `contradiction_flag` | whether the documents contradict each other | the scoring node only |
| `draft_answer` | the answer | the writing node only |
| `history` | one line per step, as a trace of the run | every node |

Three questions decide whether this form stays consistent:

1. **What happens when two nodes write the same field?** By default, the later write replaces the
   earlier one. A **reducer** can combine the two instead.
2. **What happens when they write at the same time?** Without a reducer, LangGraph refuses, because
   it cannot know which value should win.
3. **Who may write which field?** That is the **state contract**. LangGraph does not enforce it,
   so the contract is written down as data and checked.

The graph of this notebook, the skeleton of the research assistant:

```text
          +--> [collect_press] --+
START ----+                      +--> [score] --> [write_answer] --> END
          +--> [collect_news] ---+
```

The two collectors run at the same time. Each adds one document about Solvane Dynamics' quarterly
results.

## Experiment

### 1 A document in the state

Each corpus document travels in the state as a small dictionary: its identifier, title, topic,
publisher, date, authority weight, the sources it copies from (`based_on`), and its text.

In [ ]:
CORPUS = {doc.source_id: to_source_doc(doc) for doc in load_corpus()}


def source(source_id: str) -> SourceDoc:
    return CORPUS[source_id]


a1 = source("A1")
print({key: value for key, value in a1.items() if key != "text"})
print(f"text: {len(a1['text'])} characters")

### 2 The default rule: the later write replaces the earlier one

Two collectors run one after the other, in a state with no rules for combining values. Watch what
survives.

In [ ]:
class PlainState(TypedDict, total=False):
    question: str
    documents: list[SourceDoc]
    history: list[str]


def collect_press_plain(state: PlainState) -> dict:
    return {"documents": [source("A1")], "history": ["collect_press: added A1"]}


def collect_news_plain(state: PlainState) -> dict:
    return {"documents": [source("A2")], "history": ["collect_news: added A2"]}


builder = StateGraph(PlainState)
builder.add_node("collect_press", collect_press_plain)
builder.add_node("collect_news", collect_news_plain)
builder.add_edge(START, "collect_press")
builder.add_edge("collect_press", "collect_news")
builder.add_edge("collect_news", END)

plain_result = builder.compile().invoke({"question": QUESTION})
print([doc["source_id"] for doc in plain_result["documents"]])
print(plain_result["history"])

Only A2 is left, and the history no longer mentions A1. The first collector did its work, and the
second one wiped it out. There was no error and no warning. For a single answer field, replacing is
exactly right. For a list that should grow, it silently loses data.

### 3 Reducers decide how values combine

A reducer is a function that takes the current value of a field and a new update, and returns the
combined value. Two are needed:

- `history` should simply grow. Adding two lists does exactly that, so the reducer is
  `operator.add`.
- `documents` should grow too, but the same document must not appear twice, and the order should
  not depend on which node happened to finish first. That needs a small custom reducer.

In [ ]:
print(operator.add(["collect_press: added A1"], ["collect_news: added A2"]))


def merge_documents(existing: list[SourceDoc] | None, new: list[SourceDoc] | None) -> list[SourceDoc]:
    by_id: dict[str, SourceDoc] = {}
    for doc in [*(existing or []), *(new or [])]:
        by_id.setdefault(doc["source_id"], doc)
    return [by_id[source_id] for source_id in sorted(by_id)]


merged = merge_documents([source("A2")], [source("A1"), source("A2")])
print([doc["source_id"] for doc in merged])

A2 arrived twice and is kept once; the list is sorted by identifier. A reducer is attached to a
field with `Annotated[type, reducer]`. Fields without one keep the default rule, which is the right
choice for single values such as the score or the answer.

In [ ]:
class ResearchState(TypedDict, total=False):
    question: str
    documents: Annotated[list[SourceDoc], merge_documents]
    evidence_score: float
    contradiction_flag: bool
    draft_answer: str
    history: Annotated[list[str], operator.add]

### 4 The skeleton of the assistant

Four nodes, each returning only the fields it owns. Two of them are temporary stand-ins:

- the collectors add a fixed document each, instead of searching the corpus;
- the score is a plain sum of authority weights, and `contradiction_flag` is always `False`.

NB 03 replaces both with real retrieval and the evidence rules. The structure, the state and the
contract stay the same.

Two collectors start from `START`, so LangGraph runs them in the same step. The edge
`add_edge(["collect_press", "collect_news"], "score")` waits for **both** before `score` runs.

In [ ]:
def collect_press(state: ResearchState) -> dict:
    return {"documents": [source("A1")], "history": ["collect_press: added A1"]}


def collect_news(state: ResearchState) -> dict:
    return {"documents": [source("A2")], "history": ["collect_news: added A2"]}


def score(state: ResearchState) -> dict:
    documents = state["documents"]
    total = round(sum(doc["authority_weight"] for doc in documents), 2)
    return {
        "evidence_score": total,
        "contradiction_flag": False,
        "history": [f"score: {len(documents)} documents, evidence score {total}"],
    }


def write_answer(state: ResearchState) -> dict:
    ids = ", ".join(doc["source_id"] for doc in state["documents"])
    return {
        "draft_answer": (
            f"{len(state['documents'])} documents collected ({ids}), "
            f"evidence score {state['evidence_score']}."
        ),
        "history": ["write_answer: draft written"],
    }


builder = StateGraph(ResearchState)
builder.add_node("collect_press", collect_press)
builder.add_node("collect_news", collect_news)
builder.add_node("score", score)
builder.add_node("write_answer", write_answer)
builder.add_edge(START, "collect_press")
builder.add_edge(START, "collect_news")
builder.add_edge(["collect_press", "collect_news"], "score")
builder.add_edge("score", "write_answer")
builder.add_edge("write_answer", END)
graph = builder.compile()

print(graph.get_graph().draw_mermaid())

### 5 Run it

Both collectors wrote `documents` in the same step, and the reducer combined their lists. The score
reads both documents: 0.9 for the company's press release plus 0.7 for the news report.

In [ ]:
final_state = graph.invoke({"question": QUESTION})

print([doc["source_id"] for doc in final_state["documents"]])
print(final_state["evidence_score"], final_state["contradiction_flag"])
print(final_state["draft_answer"])
for line in final_state["history"]:
    print("  ", line)

Within one parallel step, the order in which the two history lines arrive is decided by LangGraph
and cannot be relied on. `documents` does not depend on that order, because `merge_documents` sorts. That is why the
reducer sorts: running the graph twice must give the same state.

`stream()` shows which node wrote which fields. This is what the contract will be checked against.

In [ ]:
updates = list(graph.stream({"question": QUESTION}))
for update in updates:
    for node, fields in update.items():
        print(f"{node:<14} wrote {sorted(fields)}")

### 6 The state contract

LangGraph lets any node write any field of the state. Nothing stops the writing node from also
changing the score. So the rules are written down as data, one entry per field with the nodes allowed
to write it, and every run is checked against them. `ANY_NODE` marks a field every node may write.
`question` is missing on purpose: only the caller sets it.

In [ ]:
ANY_NODE = "*"

CONTRACT: dict[str, set[str]] = {
    "documents": {"collect_press", "collect_news"},
    "evidence_score": {"score"},
    "contradiction_flag": {"score"},
    "draft_answer": {"write_answer"},
    "history": {ANY_NODE},
}


class ContractViolation(Exception):
    """A node wrote a state field it does not own."""


def check_contract(updates, contract: dict[str, set[str]]) -> None:
    for update in updates:
        for node, partial in update.items():
            for field_name in partial or {}:
                writers = contract.get(field_name, set())
                if ANY_NODE not in writers and node not in writers:
                    owners = ", ".join(sorted(writers)) or "the caller"
                    raise ContractViolation(
                        f"node '{node}' wrote '{field_name}', which only {owners} may write"
                    )


check_contract(updates, CONTRACT)
print("Every node kept to the contract.")

## Common mistakes

Each mistake below runs on purpose. The first one fails **silently**, which is why the cell checks
its effect with `assert`. The other two raise exactly the error named, and the test suite accepts
no other.

### Mistake 1: a node forgets to return its update

The scoring node computes the score, but the `return` line is missing. Python then returns `None`
without complaint, and for LangGraph `None` simply means "no update".

In [ ]:
def score_forgetting_return(state: ResearchState) -> None:
    total = round(sum(doc["authority_weight"] for doc in state["documents"]), 2)
    # the line "return {'evidence_score': total, ...}" is missing


forgetful = StateGraph(ResearchState)
forgetful.add_node("collect_press", collect_press)
forgetful.add_node("collect_news", collect_news)
forgetful.add_node("score", score_forgetting_return)
forgetful.add_edge(START, "collect_press")
forgetful.add_edge(START, "collect_news")
forgetful.add_edge(["collect_press", "collect_news"], "score")
forgetful.add_edge("score", END)

forgetful_result = forgetful.compile().invoke({"question": QUESTION})
print(sorted(forgetful_result))
assert "evidence_score" not in forgetful_result

The run completes, and the score is simply missing. Had `write_answer` been attached, it would have
failed with a `KeyError` on `evidence_score`: one node later, far from the actual cause. **Fix:**
every node ends with `return` and a dictionary.

A related trap is changing the state in place, for example `state["documents"].append(doc)`. It may
even look as if it worked, but it bypasses the reducers, and `stream()` never reports it. Nodes
read the state and return their changes; they never modify it directly.

### Mistake 2: two parallel nodes write a field without a reducer

The same two collectors as in the skeleton, run at the same time, but in a state where `documents`
has no reducer.

In [ ]:
class StateWithoutReducer(TypedDict, total=False):
    question: str
    documents: list[SourceDoc]
    history: Annotated[list[str], operator.add]


def collect_press_unmerged(state: StateWithoutReducer) -> dict:
    return {"documents": [source("A1")], "history": ["collect_press: added A1"]}


def collect_news_unmerged(state: StateWithoutReducer) -> dict:
    return {"documents": [source("A2")], "history": ["collect_news: added A2"]}


broken = StateGraph(StateWithoutReducer)
broken.add_node("collect_press", collect_press_unmerged)
broken.add_node("collect_news", collect_news_unmerged)
broken.add_edge(START, "collect_press")
broken.add_edge(START, "collect_news")
broken.add_edge("collect_press", END)
broken.add_edge("collect_news", END)
broken.compile().invoke({"question": QUESTION})

*"At key 'documents': Can receive only one value per step."* In experiment 2, the later write
silently won. Here both writes arrive in the same step, so there is no "later", and LangGraph
refuses rather than pick a winner at random. **Fix:** give the field a reducer with `Annotated`, as
in `ResearchState`.

This example needs its own collector functions. LangGraph reads a node's type hint
(`state: StateWithoutReducer`) to know which state it belongs to, and refuses to mix a node typed
for `ResearchState` into a graph with a different definition of `documents`.

### Mistake 3: a node writes a field it does not own

The writing node "helpfully" rounds the score to one decimal place for display, and stores it back.
LangGraph runs this without complaint. The contract check does not.

In [ ]:
def write_answer_rounding_score(state: ResearchState) -> dict:
    rounded = round(state["evidence_score"], 1)
    return {
        "draft_answer": f"Evidence score {rounded}.",
        "evidence_score": rounded,
        "history": ["write_answer: draft written"],
    }


overreaching = StateGraph(ResearchState)
overreaching.add_node("collect_press", collect_press)
overreaching.add_node("collect_news", collect_news)
overreaching.add_node("score", score)
overreaching.add_node("write_answer", write_answer_rounding_score)
overreaching.add_edge(START, "collect_press")
overreaching.add_edge(START, "collect_news")
overreaching.add_edge(["collect_press", "collect_news"], "score")
overreaching.add_edge("score", "write_answer")
overreaching.add_edge("write_answer", END)

check_contract(overreaching.compile().stream({"question": QUESTION}), CONTRACT)

From NB 04 on, the score decides where the graph goes next. A node that quietly changes it could
change the route, and nobody would see why. **Fix:** the writing node formats the score inside its
own text and never writes `evidence_score`. A field has one owner, and the contract makes the owner
visible and checkable.

## Prompt log

This notebook makes no model call: every node is plain Python, so there is no prompt to iterate.
The next prompts follow in NB 03, where a model extracts a stated figure from a document, and in
NB 04, where it rephrases a question for a second search.

## Summary

The research assistant now has its skeleton: a state with six fields, reducers that make parallel
writes safe and deterministic, and a contract that names the owner of every field. Without a
reducer, LangGraph either loses data silently (one write after another) or refuses (two writes at
once). Without a contract, nothing stops a node from changing a field it does not own.

The state, the reducer and the contract check also live in the package (`state.py`), and the
skeleton graph in `skeleton.py`. The next cell checks that the notebook and the package still agree.
The package's state keeps growing in later notebooks (NB 03 adds `claims`, for example), so the check
compares the fields this notebook defines.

In [ ]:
from research_assistant import skeleton
from research_assistant import state as package_state

package_state_result = skeleton.build_skeleton_graph().invoke({"question": QUESTION})
assert {field: package_state_result[field] for field in final_state} == final_state
assert skeleton.SKELETON_CONTRACT == CONTRACT
sample = ([source("A2")], [source("A1"), source("A2")])
assert package_state.merge_documents(*sample) == merge_documents(*sample)
print("Notebook and package agree.")

**Next: NB 03, retrieval and scoring.** The two collectors become a real search over all 21 corpus
documents, and the plain sum becomes the evidence rules: an authority floor, independent sources,
contradictions and outdated sources. The skeleton and its contract stay.

## Exercise

Add a third collector, `collect_forum`, that adds document A3: an anonymous forum post with an
authority weight of 0.2.

1. Add the node and its edges, then run `check_contract`. What else has to change, and why?
2. The plain sum now rises from 1.6 to 1.8, although the new document is an unsupported claim that
   contradicts the other two. Why is a sum of authority weights a poor measure of evidence? Which
   rule in NB 03 deals with it?